In [ ]:
import os, numpy as np, pandas as pd, torch
os.environ["WANDB_DISABLED"] = "true"  # optional

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
    DataCollatorWithPadding, EarlyStoppingCallback, EvalPrediction
)

In [ ]:
df = pd.read_csv("labeled_dataset_final.csv")
df = df[df["label"] != "No Majority"].copy()

label_map = {"Hate": 0, "Normal": 1, "Offensive": 2}
df["label"] = df["label"].map(label_map)

train_val_df, test_df = train_test_split(
    df, test_size=0.10, random_state=42, stratify=df["label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.20, random_state=42, stratify=train_val_df["label"]
)

for d in (train_df, val_df, test_df):
    d.reset_index(drop=True, inplace=True)

os.makedirs("data_splits", exist_ok=True)
test_df.to_csv("data_splits/test_data.csv", index=False)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

def drop_index_col(ds):
    cols = [c for c in ds.column_names if c == "__index_level_0__"]
    return ds.remove_columns(cols) if cols else ds

train_dataset, val_dataset, test_dataset = map(
    drop_index_col, [train_dataset, val_dataset, test_dataset]
)

def keep_text_and_label(ds):
    keep = {"text", "label"}
    remove = [c for c in ds.column_names if c not in keep]
    return ds.remove_columns(remove) if remove else ds

train_dataset, val_dataset, test_dataset = map(
    keep_text_and_label, [train_dataset, val_dataset, test_dataset]
)

In [ ]:
model_name = "cardiffnlp/twitter-roberta-base-offensive"
tokenizer  = AutoTokenizer.from_pretrained(model_name, use_fast=True)

id2label = {0: "Hate", 1: "Normal", 2: "Offensive"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,   # key line
)

model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-offensive and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
MAX_LEN = 64

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function,   batched=True)
test_dataset  = test_dataset.map(tokenize_function,  batched=True)

def drop_text(ds):
    return ds.remove_columns(["text"]) if "text" in ds.column_names else ds

train_dataset, val_dataset, test_dataset = map(
    drop_text, [train_dataset, val_dataset, test_dataset]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

Map:   0%|          | 0/4703 [00:00<?, ? examples/s]

Map:   0%|          | 0/1176 [00:00<?, ? examples/s]

Map:   0%|          | 0/654 [00:00<?, ? examples/s]

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, (list, tuple)) else eval_pred.predictions
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu().numpy()
    if logits.ndim > 2:
        logits = np.squeeze(logits)
    labels = eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": float(acc),
        "precision": float(p_w), "recall": float(r_w), "f1": float(f1_w),
        "precision_macro": float(p_m), "recall_macro": float(r_m), "f1_macro": float(f1_m),
    }

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

training_args = TrainingArguments(
    output_dir="./roberta_offensive_finetuned_3class",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    fp16=(not use_bf16),
    bf16=use_bf16,
    logging_dir="./logs",
    logging_steps=50,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    report_to="none",
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
    class_weights=class_weights,
)

torch.cuda.empty_cache()
trainer.train()

trainer.save_model("./roberta_offensive_finetuned_3class")
tokenizer.save_pretrained("./roberta_offensive_finetuned_3class")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro
1,0.626100,0.576100,0.758503,0.763188,0.758503,0.757811,0.687408,0.709230,0.695383
2,0.508600,0.726560,0.750000,0.759034,0.750000,0.747294,0.683836,0.704341,0.688183


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


('./roberta_offensive_finetuned_3class/tokenizer_config.json',
 './roberta_offensive_finetuned_3class/special_tokens_map.json',
 './roberta_offensive_finetuned_3class/vocab.json',
 './roberta_offensive_finetuned_3class/merges.txt',
 './roberta_offensive_finetuned_3class/added_tokens.json',
 './roberta_offensive_finetuned_3class/tokenizer.json')

In [ ]:
raw = trainer.predict(test_dataset)
logits = raw.predictions[0] if isinstance(raw.predictions, (list, tuple)) else raw.predictions
if isinstance(logits, torch.Tensor):
    logits = logits.detach().cpu().numpy()
if logits.ndim > 2:
    logits = np.squeeze(logits)

labels = raw.label_ids
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds = np.argmax(logits, axis=-1)

print("\n=== Test set metrics ===")
print(compute_metrics(EvalPrediction(predictions=logits, label_ids=labels)))

print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(labels, preds, labels=[0,1,2]))

print("\nClassification report:")
print(classification_report(labels, preds, labels=[0,1,2],
                            target_names=["Hate","Normal","Offensive"], digits=4))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



=== Test set metrics ===
{'accuracy': 0.7935779816513762, 'precision': 0.7994587238004313, 'recall': 0.7935779816513762, 'f1': 0.7930360725700323, 'precision_macro': 0.7300486867817044, 'recall_macro': 0.7552952429473349, 'f1_macro': 0.7392238360609813}

Confusion matrix (rows=true, cols=pred):
[[ 28   4  11]
 [  0 258  33]
 [ 22  65 233]]

Classification report:
              precision    recall  f1-score   support

        Hate     0.5600    0.6512    0.6022        43
      Normal     0.7890    0.8866    0.8350       291
   Offensive     0.8412    0.7281    0.7806       320

    accuracy                         0.7936       654
   macro avg     0.7300    0.7553    0.7392       654
weighted avg     0.7995    0.7936    0.7930       654



In [ ]:
out = pd.DataFrame({
    "text": test_df["text"],
    "true_label_id": labels,
    "true_label": [id2label[int(t)] for t in labels],
    "pred_label_id": preds,
    "pred_label": [id2label[int(p)] for p in preds],
    "prob_Hate": probs[:,0],
    "prob_Normal": probs[:,1],
    "prob_Offensive": probs[:,2],
})
out.to_csv("test_predictions.csv", index=False)

prec, rec, f1, sup = precision_recall_fscore_support(labels, preds, labels=[0,1,2], zero_division=0)
pd.DataFrame({
    "label": ["Hate","Normal","Offensive"],
    "precision": prec, "recall": rec, "f1": f1, "support": sup
}).to_csv("per_class_metrics.csv", index=False)

print("\nSaved: test_predictions.csv, per_class_metrics.csv, and model folder roberta_offensive_finetuned_3class")


Saved: test_predictions.csv, per_class_metrics.csv, and model folder roberta_offensive_finetuned_3class


In [ ]:
def predict(texts):
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    labels_pred = [id2label[int(i)] for i in probs.argmax(1)]
    return labels_pred, probs